# ML0 — Baseline limpio del subapartado O4

**Objetivo**: predecir la celda donde estará la gaviota al día siguiente usando Random Forest, XGBoost y LightGBM. Baseline construido desde cero ignorando la línea ML1–ML6 existente para evitar sesgo de confirmación.

**Entrada**: `data/processed/hmm5.csv` (21 081 filas, versión canónica del HMM en el TFG; ver `docs/O3_hmm.md` sección "Decisión final"). El `estado_hmm` proviene de HMM5 (4 features recomendadas por la tutora: step + turning + vegetación + horas de luz).
**Salida**: tabla comparativa de los 3 modelos + `data/processed/ml0_predictions.csv` con top-5 celdas predichas por fila de test.

**Decisión clave** (ver `O4_plan.md` §1.1): las columnas `step_length`, `bearing` y `turning_angle` de `hmm5.csv` describen el salto **t→t+1** y serían leakage para predecir `cell(t+1)`. Se recalculan como salto previo (t-1→t): `step_prev`, `bearing_prev`. La primera fila de cada trayectoria se descarta (sin t-1) además de la última (sin target).

**Nota sobre `horas_luz` / `daylight_h`**: `hmm5.csv` ya contiene la columna `horas_luz` calculada con la fórmula de Spencer 1971. ML0 conserva su propio cálculo `daylight_h` con `ephem.next_rising/setting` (más preciso, valida con asserts) para mantener la independencia metodológica respecto al HMM. Ambos valores deben ser muy similares (verificable como sanity check).

## 1. Imports y constantes globales

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
from functools import lru_cache

import ephem
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
GRID_RES = 0.5            # grados; igual a markov1.ipynb
TOP_K = (1, 3, 5)

# Presupuesto por defecto = prototipo. Para resultados publicables del TFG:
#   N_ITER_RANDOM = 30, CV_SPLITS = 5  (~7-8 h en sesión nocturna).
N_ITER_RANDOM = 3
CV_SPLITS = 3

np.random.seed(RANDOM_STATE)
print(f'Configuración: N_ITER_RANDOM={N_ITER_RANDOM}, CV_SPLITS={CV_SPLITS}')


### Utilidades reutilizadas

`haversine_km` y `calculate_bearing` se reproducen literalmente desde `notebooks/HMM.ipynb` (celda 0) — funciones puras y validadas.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Distancia haversine entre dos puntos (lat, lon) en grados, en km."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def calculate_bearing(lat1, lon1, lat2, lon2):
    """Rumbo great-circle inicial en grados [0, 360); 0=norte, sentido horario."""
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dlambda = np.radians(lon2 - lon1)
    y = np.sin(dlambda) * np.cos(phi2)
    x = np.cos(phi1) * np.sin(phi2) - np.sin(phi1) * np.cos(phi2) * np.cos(dlambda)
    return (np.degrees(np.arctan2(y, x)) + 360) % 360


## 2. Carga del CSV y EDA mínimo

In [ ]:
df = pd.read_csv('../data/processed/hmm5.csv', parse_dates=['date'])
df = df.sort_values(['trayectoria_id', 'date']).reset_index(drop=True)

expected_cols = {
    'animal_id', 'date', 'hora', 'lon', 'lat', 'veg_low', 'veg_high',
    'trayectoria_id', 'step_length', 'bearing', 'turning_angle',
    'horas_luz', 'estado_hmm',
}
assert set(df.columns) == expected_cols, f'Columnas inesperadas: {set(df.columns) ^ expected_cols}'
assert df.shape == (21081, 13), f'Esperado (21081, 13), encontrado {df.shape}'
assert df.isna().sum().sum() == 0, 'NaN inesperados en hmm5.csv'

print(f'Filas: {len(df):,}')
print(f'Aves únicas: {df.animal_id.nunique()}')
print(f'Trayectorias únicas: {df.trayectoria_id.nunique()}')
print(f'Rango fechas: {df.date.min().date()} → {df.date.max().date()}')
print('\nDistribución de estado_hmm (HMM5 canónico):')
print(df.estado_hmm.value_counts(normalize=True).rename({0: 'migración', 1: 'estacionario'}).round(3))

## 3. Discretización del espacio en celdas

Grid regular de 0,5° anclado al mínimo del CSV — coherente con `markov1.ipynb`.

In [ ]:
lon_min = df['lon'].min()
lat_min = df['lat'].min()

df['grid_x'] = ((df['lon'] - lon_min) / GRID_RES).astype(int)
df['grid_y'] = ((df['lat'] - lat_min) / GRID_RES).astype(int)
df['cell_id'] = df['grid_x'].astype(str) + '_' + df['grid_y'].astype(str)

n_celdas = df['cell_id'].nunique()
print(f'Anclaje: lon_min={lon_min:.4f}, lat_min={lat_min:.4f}')
print(f'Celdas únicas: {n_celdas}')
assert 1100 <= n_celdas <= 1300, 'n_celdas fuera del rango esperado [1100, 1300]'


## 4. Target + features previas (corrección del leakage)

Por trayectoria se calculan:
- `target_cell = cell_id.shift(-1)` — celda donde estará el ave mañana.
- `step_prev`, `bearing_prev` con `(pos_{t-1} → pos_t)` — observables HOY, sin futuro.
- `turning_prev` = diff(bearing_prev), normalizado a [-π, π].

Se descartan la primera (sin t-1) y la última (sin target) fila de cada trayectoria.

In [ ]:
df['next_cell'] = df.groupby('trayectoria_id')['cell_id'].shift(-1)
df['prev_lat'] = df.groupby('trayectoria_id')['lat'].shift(1)
df['prev_lon'] = df.groupby('trayectoria_id')['lon'].shift(1)

df['step_prev'] = haversine_km(df['prev_lat'], df['prev_lon'], df['lat'], df['lon'])
df['bearing_prev'] = calculate_bearing(df['prev_lat'], df['prev_lon'], df['lat'], df['lon'])
delta = df.groupby('trayectoria_id')['bearing_prev'].diff()
df['turning_prev'] = np.arctan2(np.sin(np.radians(delta)), np.cos(np.radians(delta)))

filas_antes = len(df)
df = df.dropna(subset=['next_cell', 'step_prev', 'bearing_prev']).reset_index(drop=True)
filas_descartadas = filas_antes - len(df)
print(f'Filas descartadas (primera + última de cada trayectoria): {filas_descartadas:,}')
print(f'Filas válidas para entrenamiento: {len(df):,}')

df = df.rename(columns={'next_cell': 'target_cell'})

# Sanity check del fix de leakage:
centro_real_lat = lat_min + (df['target_cell'].str.split('_').str[1].astype(int) + 0.5) * GRID_RES
centro_real_lon = lon_min + (df['target_cell'].str.split('_').str[0].astype(int) + 0.5) * GRID_RES
dist_destino = haversine_km(df['lat'].values, df['lon'].values, centro_real_lat.values, centro_real_lon.values)
corr_orig = pd.Series(df['step_length'].values).corr(pd.Series(dist_destino))
corr_prev = pd.Series(df['step_prev'].values).corr(pd.Series(dist_destino))
print(f'\nCorrelación con dist(pos_t, centro_target):')
print(f'  step_length original (t→t+1, leakage): {corr_orig:.3f}')
print(f'  step_prev recalculado (t-1→t, OK):     {corr_prev:.3f}')
assert corr_prev < 0.7, 'step_prev sigue demasiado correlacionado con destino — revisar'


## 5. Derivación de features restantes

### 5.1 Mes circular

In [ ]:
df['mes_num'] = df['date'].dt.month
df['sin_mes'] = np.sin(2 * np.pi * df['mes_num'] / 12)
df['cos_mes'] = np.cos(2 * np.pi * df['mes_num'] / 12)

assert np.allclose(df['sin_mes'] ** 2 + df['cos_mes'] ** 2, 1.0)
print(f'Meses presentes: {sorted(df.mes_num.unique())}')


### 5.2 Bearing circular (sobre el rumbo previo)

In [ ]:
df['sin_bearing'] = np.sin(np.radians(df['bearing_prev']))
df['cos_bearing'] = np.cos(np.radians(df['bearing_prev']))
assert np.allclose(df['sin_bearing'] ** 2 + df['cos_bearing'] ** 2, 1.0)


### 5.3 Horas de luz diarias (`daylight_h`)

Calculadas con `ephem.Observer.next_rising/setting(ephem.Sun())`. Se cachean por `(lat redondeada a 0,5°, fecha)` para reducir llamadas.

In [ ]:
@lru_cache(maxsize=None)
def _daylight_h_cached(lat_round, date_iso):
    obs = ephem.Observer()
    obs.lat = str(lat_round)
    obs.lon = '0'
    obs.date = date_iso
    obs.horizon = '0'
    obs.pressure = 0
    sun = ephem.Sun()
    try:
        rise = obs.next_rising(sun)
        set_ = obs.next_setting(sun)
        hours = (set_ - rise) * 24.0
        if hours < 0:
            rise = obs.previous_rising(sun)
            hours = (set_ - rise) * 24.0
        return float(hours)
    except ephem.NeverUpError:
        return 0.0
    except ephem.AlwaysUpError:
        return 24.0


def daylight_h(lat, date):
    return _daylight_h_cached(round(float(lat) * 2) / 2, date.strftime('%Y-%m-%d'))


t0 = time.time()
df['daylight_h'] = [daylight_h(la, da) for la, da in zip(df['lat'], df['date'])]
print(f'daylight_h calculado en {time.time() - t0:.1f}s')
print(f'cache info: {_daylight_h_cached.cache_info()}')

df_alta_lat = df[df['lat'] > 50]
if len(df_alta_lat) > 0:
    medias_mes = df_alta_lat.groupby('mes_num')['daylight_h'].mean().round(2)
    print('\nHoras de luz medias por mes en lat > 50°N:')
    print(medias_mes)
    assert medias_mes.idxmax() in [5, 6, 7], 'Máximo de daylight_h debería estar en mayo–julio'
    assert medias_mes.idxmin() in [11, 12, 1], 'Mínimo de daylight_h debería estar en nov–enero'


### 5.4 Lista final de features

In [ ]:
FEATURES = [
    'grid_x', 'grid_y',
    'mes_num', 'sin_mes', 'cos_mes',
    'step_prev',
    'sin_bearing', 'cos_bearing',
    'veg_low', 'veg_high',
    'daylight_h',
    'estado_hmm',
]
print(f'{len(FEATURES)} features:')
for f in FEATURES:
    print(f'  - {f}')


## 6. Split por animal cronológico (80 / 20)

Convención del proyecto: dentro de cada `animal_id`, primer 80 % cronológico → train, último 20 % → test. Animales con menos de 5 observaciones se omiten.

In [ ]:
train_parts, test_parts = [], []
n_omitidos = 0
for animal, grupo in df.groupby('animal_id'):
    grupo = grupo.sort_values('date')
    if len(grupo) < 5:
        n_omitidos += 1
        continue
    n_train = int(len(grupo) * 0.8)
    train_parts.append(grupo.iloc[:n_train])
    test_parts.append(grupo.iloc[n_train:])

df_train = pd.concat(train_parts).reset_index(drop=True)
df_test = pd.concat(test_parts).reset_index(drop=True)

for animal in df_train['animal_id'].unique():
    if animal in df_test['animal_id'].values:
        max_tr = df_train[df_train.animal_id == animal]['date'].max()
        min_te = df_test[df_test.animal_id == animal]['date'].min()
        assert max_tr <= min_te, f'Leakage temporal en animal {animal}'

print(f'Animales omitidos (<5 obs.): {n_omitidos}')
print(f'Train: {len(df_train):,} filas | {df_train.animal_id.nunique()} aves')
print(f'Test:  {len(df_test):,} filas | {df_test.animal_id.nunique()} aves')


## 7. Encoding del target

`LabelEncoder` ajustado **solo** sobre `target_cell` de train. Filas de test con celdas no vistas en train → descartar.

In [ ]:
le = LabelEncoder()
le.fit(df_train['target_cell'])
n_classes = len(le.classes_)

celdas_train = set(le.classes_)
mask_test_valido = df_test['target_cell'].isin(celdas_train)
n_descartadas = (~mask_test_valido).sum()
pct_descartadas = 100 * n_descartadas / len(df_test)
df_test = df_test[mask_test_valido].reset_index(drop=True)

X_train = df_train[FEATURES].values
y_train = le.transform(df_train['target_cell'])
X_test = df_test[FEATURES].values
y_test = le.transform(df_test['target_cell'])
groups_train = df_train['animal_id'].values

print(f'Clases (celdas en train): {n_classes}')
print(f'Filas test descartadas: {n_descartadas:,} ({pct_descartadas:.2f} %)')
if pct_descartadas > 15:
    print('AVISO: más del 15% del test descartado por celdas no vistas en train.')
print(f'X_train shape: {X_train.shape} | X_test shape: {X_test.shape}')


## 8. Esquema de validación cruzada

`GroupKFold` por `animal_id`: cada fold deja un subconjunto de animales fuera. Mide capacidad de generalizar a animales no vistos — más exigente que el split final por periodo, pero coherente con la estructura del problema.

In [ ]:
cv = GroupKFold(n_splits=CV_SPLITS)

for fold_idx, (tr, va) in enumerate(cv.split(X_train, y_train, groups=groups_train)):
    set_tr = set(groups_train[tr])
    set_va = set(groups_train[va])
    assert set_tr & set_va == set(), 'GroupKFold mezcló animales entre folds'
    if fold_idx == 0:
        print(f'fold 0: {len(set_tr)} animales train | {len(set_va)} animales val')
print(f'GroupKFold OK: {CV_SPLITS} folds, sin solapamiento de animales.')


## 9. Tuning + entrenamiento de los tres modelos

`RandomizedSearchCV` con `accuracy` como scorer del CV. Las métricas top-3, top-5 y haversine se calculan solo en el test final.

**Memoria** (15 GB RAM): con ~1100-1300 clases (celdas), los clasificadores multiclase consumen mucha RAM por proceso. Por eso se fuerza `n_jobs=1` y `pre_dispatch=1` en el search y `n_jobs=1` dentro del modelo: los fits se ejecutan secuencialmente y nunca hay más de un estimador en memoria. Las distribuciones de hiperparámetros se han recortado (menos `n_estimators`, `min_samples_leaf` ≥ 5, sin `class_weight='balanced'`, `num_leaves` ≤ 63) para no saturar memoria con tantas clases.

### 9.1 Random Forest

In [ ]:
rf_param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 'log2', 0.3],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_distributions=rf_param_dist,
    n_iter=N_ITER_RANDOM,
    cv=cv,
    scoring='accuracy',
    n_jobs=1,
    pre_dispatch=1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=1,
)

t0 = time.time()
rf_search.fit(X_train, y_train, groups=groups_train)
print(f'\nTiempo total RF: {(time.time() - t0) / 60:.1f} min')
print(f'Mejor CV accuracy: {rf_search.best_score_:.4f}')
print(f'Mejores parámetros: {rf_search.best_params_}')
rf_best = rf_search.best_estimator_

### 9.2 XGBoost

In [ ]:
class XGBRemap(xgb.XGBClassifier):
    """XGBClassifier que re-encodea y a [0..k-1] por fit y mapea de vuelta al
    predecir. Necesario porque GroupKFold deja fuera animales y por tanto
    algunas celdas (clases) no aparecen en el fold de train; XGBoost exige
    que np.unique(y) sea contiguo desde 0. En el refit final con todo y_train
    el remap es identidad (k = n_classes), así que predict_proba sobre test
    devuelve columnas alineadas con la codificación global."""

    def fit(self, X, y, **kwargs):
        self._fold_le = LabelEncoder()
        y_local = self._fold_le.fit_transform(y)
        super().fit(X, y_local, **kwargs)
        return self

    def predict(self, X, **kwargs):
        y_local = super().predict(X, **kwargs)
        return self._fold_le.inverse_transform(y_local.astype(int))


xgb_param_dist = {
    'n_estimators': [200, 300, 400],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 5, 10],
}

xgb_base = XGBRemap(
    objective='multi:softprob',
    tree_method='hist',
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0,
)

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=N_ITER_RANDOM,
    cv=cv,
    scoring='accuracy',
    n_jobs=1,
    pre_dispatch=1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=1,
)

t0 = time.time()
xgb_search.fit(X_train, y_train, groups=groups_train)
print(f'\nTiempo total XGB: {(time.time() - t0) / 60:.1f} min')
print(f'Mejor CV accuracy: {xgb_search.best_score_:.4f}')
print(f'Mejores parámetros: {xgb_search.best_params_}')
xgb_best = xgb_search.best_estimator_

### 9.3 LightGBM

In [ ]:
class LGBMRemap(lgb.LGBMClassifier):
    """LGBMClassifier que re-encodea y a [0..k-1] por fit y mapea de vuelta al
    predecir. Mismo motivo que XGBRemap: con GroupKFold sobre 1100+ celdas,
    los folds de train tienen huecos en el conjunto de clases. Sin num_class
    fijado, LightGBM lo infiere por fold; en refit final coincide con n_classes."""

    def fit(self, X, y, **kwargs):
        self._fold_le = LabelEncoder()
        y_local = self._fold_le.fit_transform(y)
        super().fit(X, y_local, **kwargs)
        return self

    def predict(self, X, **kwargs):
        y_local = super().predict(X, **kwargs)
        return self._fold_le.inverse_transform(y_local.astype(int))


lgb_param_dist = {
    'n_estimators': [200, 300, 400],
    'num_leaves': [31, 63],
    'max_depth': [8, 12, -1],
    'learning_rate': [0.03, 0.05, 0.1],
    'min_child_samples': [20, 50, 100],
    'feature_fraction': [0.7, 0.85, 1.0],
}

lgb_base = LGBMRemap(
    objective='multiclass',
    boosting_type='gbdt',
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=-1,
)

lgb_search = RandomizedSearchCV(
    lgb_base,
    param_distributions=lgb_param_dist,
    n_iter=N_ITER_RANDOM,
    cv=cv,
    scoring='accuracy',
    n_jobs=1,
    pre_dispatch=1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=1,
)

t0 = time.time()
lgb_search.fit(X_train, y_train, groups=groups_train)
print(f'\nTiempo total LGBM: {(time.time() - t0) / 60:.1f} min')
print(f'Mejor CV accuracy: {lgb_search.best_score_:.4f}')
print(f'Mejores parámetros: {lgb_search.best_params_}')
lgb_best = lgb_search.best_estimator_

## 10. Evaluación en el test

Por modelo: top-1, top-3, top-5 accuracy, haversine mediana al centroide real, % predicciones a ≤ 50 km y ≤ 100 km. Todo se desglosa además por `estado_hmm`.

In [ ]:
def centroide(grid_x, grid_y):
    lon_c = lon_min + (grid_x + 0.5) * GRID_RES
    lat_c = lat_min + (grid_y + 0.5) * GRID_RES
    return lat_c, lon_c


def topk_accuracy(probs, y_true, k):
    top = np.argpartition(-probs, kth=min(k - 1, probs.shape[1] - 1), axis=1)[:, :k]
    return float(np.mean([y_true[i] in top[i] for i in range(len(y_true))]))


def evaluar_modelo(name, model, X_test, y_test, df_test, le):
    proba = model.predict_proba(X_test)
    y_pred = np.argmax(proba, axis=1)

    cells_pred = le.inverse_transform(y_pred)
    cells_real = le.inverse_transform(y_test)
    grid_pred = pd.Series(cells_pred).str.split('_').apply(lambda p: (int(p[0]), int(p[1])))
    grid_real = pd.Series(cells_real).str.split('_').apply(lambda p: (int(p[0]), int(p[1])))
    lat_pred = np.array([centroide(gx, gy)[0] for gx, gy in grid_pred])
    lon_pred = np.array([centroide(gx, gy)[1] for gx, gy in grid_pred])
    lat_real = np.array([centroide(gx, gy)[0] for gx, gy in grid_real])
    lon_real = np.array([centroide(gx, gy)[1] for gx, gy in grid_real])
    dist_km = haversine_km(lat_pred, lon_pred, lat_real, lon_real)

    out = {'modelo': name}
    for k in TOP_K:
        out[f'top{k}_global'] = topk_accuracy(proba, y_test, k)
    out['hav_med_global'] = float(np.median(dist_km))
    out['p50km_global'] = float(np.mean(dist_km <= 50))
    out['p100km_global'] = float(np.mean(dist_km <= 100))

    for estado, etiqueta in [(0, 'mig'), (1, 'est')]:
        m = df_test['estado_hmm'].values == estado
        if m.sum() == 0:
            continue
        for k in TOP_K:
            out[f'top{k}_{etiqueta}'] = topk_accuracy(proba[m], y_test[m], k)
        out[f'hav_med_{etiqueta}'] = float(np.median(dist_km[m]))
        out[f'p50km_{etiqueta}'] = float(np.mean(dist_km[m] <= 50))
        out[f'p100km_{etiqueta}'] = float(np.mean(dist_km[m] <= 100))
    out['_proba'] = proba
    out['_dist_km'] = dist_km
    return out


In [ ]:
resultados = {
    'RF': evaluar_modelo('Random Forest', rf_best, X_test, y_test, df_test, le),
    'XGB': evaluar_modelo('XGBoost', xgb_best, X_test, y_test, df_test, le),
    'LGBM': evaluar_modelo('LightGBM', lgb_best, X_test, y_test, df_test, le),
}

for nombre, r in resultados.items():
    print(f'\n=== {r["modelo"]} ===')
    print(f'  Top-1 global: {r["top1_global"]:.4f}  (mig {r.get("top1_mig", float("nan")):.4f} | est {r.get("top1_est", float("nan")):.4f})')
    print(f'  Top-3 global: {r["top3_global"]:.4f}  (mig {r.get("top3_mig", float("nan")):.4f} | est {r.get("top3_est", float("nan")):.4f})')
    print(f'  Top-5 global: {r["top5_global"]:.4f}  (mig {r.get("top5_mig", float("nan")):.4f} | est {r.get("top5_est", float("nan")):.4f})')
    print(f'  Haversine mediana (km): {r["hav_med_global"]:.1f}  (mig {r.get("hav_med_mig", float("nan")):.1f} | est {r.get("hav_med_est", float("nan")):.1f})')
    print(f'  ≤ 50 km : {r["p50km_global"]:.4f}  (mig {r.get("p50km_mig", float("nan")):.4f} | est {r.get("p50km_est", float("nan")):.4f})')
    print(f'  ≤ 100 km: {r["p100km_global"]:.4f}  (mig {r.get("p100km_mig", float("nan")):.4f} | est {r.get("p100km_est", float("nan")):.4f})')


## 11. Importancia de features (los 3 modelos)

Comparativa de `feature_importances_`. Útil para validar empíricamente el caveat sobre `veg_low/veg_high` (que HMM3 demostró inocuas) y `daylight_h`.

In [ ]:
lgb_imp = lgb_best.feature_importances_.astype(float)
if lgb_imp.sum() == 0:
    lgb_imp = np.ones_like(lgb_imp)
imp_df = pd.DataFrame({
    'feature': FEATURES,
    'RF': rf_best.feature_importances_,
    'XGB': xgb_best.feature_importances_,
    'LGBM': lgb_imp / lgb_imp.sum(),
}).set_index('feature')

imp_norm = imp_df.div(imp_df.sum(axis=0), axis=1).round(4)
imp_norm['media'] = imp_norm.mean(axis=1).round(4)
print('Importancia de features (normalizada por modelo, suma columna = 1):')
print(imp_norm.sort_values('media', ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, modelo in zip(axes, ['RF', 'XGB', 'LGBM']):
    imp_norm[modelo].sort_values().plot.barh(ax=ax, color='tab:blue')
    ax.set_title(modelo)
    ax.set_xlabel('importancia normalizada')
plt.tight_layout()
plt.show()


## 12. Tabla comparativa final

In [ ]:
filas = []
for nombre, r in resultados.items():
    fila = {
        'modelo': r['modelo'],
        'Top-1 global': f'{r["top1_global"] * 100:.2f} %'.replace('.', ','),
        'Top-3 global': f'{r["top3_global"] * 100:.2f} %'.replace('.', ','),
        'Top-5 global': f'{r["top5_global"] * 100:.2f} %'.replace('.', ','),
        'Top-1 mig': f'{r.get("top1_mig", float("nan")) * 100:.2f} %'.replace('.', ','),
        'Top-1 est': f'{r.get("top1_est", float("nan")) * 100:.2f} %'.replace('.', ','),
        'Hav. med. (km)': f'{r["hav_med_global"]:.1f}'.replace('.', ','),
        '≤ 50 km': f'{r["p50km_global"] * 100:.2f} %'.replace('.', ','),
        '≤ 100 km': f'{r["p100km_global"] * 100:.2f} %'.replace('.', ','),
    }
    filas.append(fila)

tabla = pd.DataFrame(filas).set_index('modelo')
print(tabla.to_string())

ganador = max(resultados, key=lambda k: resultados[k]['top1_global'])
print(f'\nModelo ganador por Top-1 global: {resultados[ganador]["modelo"]}')

### 12.1 Persistencia de predicciones (input para O5)

In [ ]:
ganador_proba = resultados[ganador]['_proba']
top5_idx = np.argsort(-ganador_proba, axis=1)[:, :5]
top5_cells = np.vectorize(lambda i: le.classes_[i])(top5_idx)
top5_probs = np.take_along_axis(ganador_proba, top5_idx, axis=1)

out = pd.DataFrame({
    'animal_id': df_test['animal_id'].values,
    'date': df_test['date'].dt.strftime('%Y-%m-%d').values,
    'cell_actual': df_test['cell_id'].values,
    'cell_real': df_test['target_cell'].values,
    'estado_hmm': df_test['estado_hmm'].values,
})
for k in range(5):
    out[f'top{k+1}_cell'] = top5_cells[:, k]
    out[f'top{k+1}_prob'] = top5_probs[:, k].round(4)
out['modelo_ganador'] = resultados[ganador]['modelo']

out.to_csv('../data/processed/ml0_predictions.csv', index=False)
print(f'Predicciones guardadas en data/processed/ml0_predictions.csv ({len(out):,} filas)')


## 13. Conclusiones provisionales

_Las conclusiones detalladas se documentarán en `notebooks/O4_plan.md` §1.10–1.12 (Resultados, Lecciones aprendidas, Sugerencias para variantes)._